In [ ]:
# ============================================================
# Convert PDF to Markdown
# ===========================================================

!python convert_pdfs_to_markdown.py --output-dir data/markdown --manifest-path data/markdown_manifest.json --force

In [1]:
from pipeline_helpers import load_or_build_markdown_nodes

PIPELINE_CHECKPOINT_DIR = "./checkpoints/pipeline"
FORCE_REBUILD_NODES = False

documents, nodes = load_or_build_markdown_nodes(
    input_dir="./data/markdown_filtered",
    checkpoint_dir=PIPELINE_CHECKPOINT_DIR,
    force_rebuild=FORCE_REBUILD_NODES,
)

print(f"--- Division Complete ---")
print(f"Total logical sections found: {len(nodes)}\n")

C:\Users\derrm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


Loaded cached documents/nodes from checkpoints\pipeline (2047 nodes from 1 files).
--- Division Complete ---
Total logical sections found: 2047



In [2]:
# ============================================================
# NEO4J CONNECTION SETUP (imported from external module)
# ============================================================

from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path(".env"), override=False)

from neo4j_helpers import create_driver, test_connection

driver = create_driver()
test_connection(driver)

Neo4j connection: connected
URI: neo4j+s://90eb030b.databases.neo4j.io
Database: 90eb030b


True

In [3]:
# ============================================================
# NEO4J QUERY HELPERS (imported from external module)
# ============================================================

from neo4j_helpers import (
    neo4j_forward_lookup,
    neo4j_reverse_lookup,
    neo4j_comparison,
    neo4j_text_query,
    neo4j_get_all_entities,
    neo4j_get_all_relations,
    neo4j_get_entity_neighbors,
)

print("Neo4j query helpers imported")

Neo4j query helpers imported


In [4]:
nodes[0].text

'## **Diagnostic Criteria and Codes**\r\n\r\nNeurodevelopmental Disorders Schizophrenia Spectrum and Other Psychotic Disorders Bipolar and Related Disorders Depressive Disorders Anxiety Disorders Obsessive-Compulsive and Related Disorders Trauma- and Stressor-Related Disorders Dissociative Disorders Somatic Symptom and Related Disorders Feeding and Eating Disorders Elimination Disorders Sleep-Wake Disorders Sexual Dysfunctions Gender Dysphoria Disruptive, Impulse-Control, and Conduct Disorders Substance-Related and Addictive Disorders Neurocognitive Disorders Personality Disorders Paraphilic Disorders Other Mental Disorders and Additional Codes \r\n\r\nMedication-Induced Movement Disorders and Other Adverse Effects of Medication Other Conditions That May Be a Focus of Clinical Attention \r\n\r\nThis section contains the diagnostic criteria approved for routine clinical use along with ICD-10-CM diagnostic codes. For each mental disorder, the diagnostic criteria are followed by descripti

In [5]:
from neo4j_helpers import restore_from_neo4j

(
    custom_entities,
    entity_id_to_node,
    custom_relations,
    relation_metadata,
    entity_descriptions,
    entity_sources,
) = restore_from_neo4j(driver)

✅ Restored from Neo4j:
   Database:  90eb030b
   Entities:  323
   Relations: 333


In [4]:
# API KEY + enrichment model

import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path(".env"), override=False)

api_key = os.getenv("OPENAI_API_KEY")
ENRICH_MODEL = os.getenv("PIPELINE_ENRICH_MODEL", "gpt-4o-mini")
ENRICH_REASONING_EFFORT = os.getenv("PIPELINE_ENRICH_REASONING_EFFORT", "low")
ENRICH_MAX_TOKENS = int(os.getenv("PIPELINE_ENRICH_MAX_TOKENS", "60"))

# LLM initialization

from llama_index.llms.openai import OpenAI

llm = OpenAI(
    model=ENRICH_MODEL,
    api_key=api_key,
    temperature=0,
    max_tokens=ENRICH_MAX_TOKENS,
    reasoning_effort=ENRICH_REASONING_EFFORT,
)

print(f"Enrichment model: {ENRICH_MODEL}")
print(f"Reasoning effort: {ENRICH_REASONING_EFFORT}")
print(f"Max output tokens: {ENRICH_MAX_TOKENS}")


C:\Users\derrm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


Enrichment model: gpt-4o-mini
Reasoning effort: low
Max output tokens: 60


In [7]:
# ============================================================
# Nodes enrichment
# ============================================================
from pipeline_helpers import enrich_nodes_with_context

FORCE_REBUILD_ENRICHMENT = False

enriched_nodes = enrich_nodes_with_context(
    nodes,
    llm_client=llm,
    checkpoint_dir=PIPELINE_CHECKPOINT_DIR,
    resume=True,
    force_rebuild=FORCE_REBUILD_ENRICHMENT,
    save_every=25,
    progress_every=25,
)


Loaded completed enrichment checkpoint from checkpoints\pipeline (2047/2047 nodes, model=gpt-4o-mini).


In [ ]:
# # ============================================================
# # Full DSM GraphRAG run (start from scratch)
# # ============================================================
# !python pipeline.py --node-checkpoint-dir .\checkpoints\pipeline --graph-checkpoint-dir .\checkpoints\dsm_graph_v6 --community-checkpoint-dir .\checkpoints\dsm_graph_v6 --force-rebuild-graph --force-rebuild-communities --progress-every 10 --smoke-test

In [1]:
# ============================================================
# Full DSM GraphRAG run (resumable)
# ============================================================
import os
os.environ["PYTHONUNBUFFERED"] = "1"

!python pipeline.py --node-checkpoint-dir .\\checkpoints\\pipeline --graph-checkpoint-dir .\\checkpoints\\dsm_graph_v6 --community-checkpoint-dir .\\checkpoints\\dsm_graph_v6 --progress-every 1 --smoke-test



STAGE 1: MARKDOWN NODES
./data/markdown_filtered
Loaded cached documents/nodes from checkpoints\pipeline (2047 nodes from 1 files).
Loaded 1 documents and 2047 markdown nodes.

STAGE 2: ENRICH NODES
checkpoint: .\\checkpoints\\pipeline
Loaded completed enrichment checkpoint from checkpoints\pipeline (2047/2047 nodes, model=gpt-4o-mini).
Prepared 2047 enriched nodes.

STAGE 3: EXTRACT GRAPH
checkpoint: .\\checkpoints\\dsm_graph_v6
Resuming graph extraction from chunk 1226/2047 using partial checkpoint in checkpoints\dsm_graph_v6.
[GRAPH] 1225/2047 ( 59.8%) | elapsed 00:00:00 | resume point
[GRAPH] 1226/2047 ( 59.9%) | elapsed 00:00:17 | rate 0.06/s | eta 03:47:21 | chunk 62c64f93 | 2273 entities | 2633 relations
[GRAPH] 1227/2047 ( 59.9%) | elapsed 00:00:30 | rate 0.07/s | eta 03:24:08 | chunk addfa6d2 | 2273 entities | 2633 relations
[GRAPH] 1228/2047 ( 60.0%) | elapsed 00:00:35 | rate 0.09/s | eta 02:38:27 | chunk 3b72737c | 2274 entities | 2635 relations
[GRAPH] 1229/2047 ( 60.0%) |

2026-04-09 11:21:52.060425: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 11:21:59.906737: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
C:\Users\derrm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can on

In [3]:
# ====================================================================
# Query command
# ====================================================================
!python pipeline.py --graph-checkpoint-dir .\checkpoints\dsm_graph_v6 --clean-graph-checkpoint-dir .\checkpoints\dsm_graph_v6_clean --community-checkpoint-dir .\checkpoints\dsm_graph_v6_clean --query "What are common comorbidities of language disorder?"


^C



STAGE 1: LOAD GRAPH CHECKPOINT
.\checkpoints\dsm_graph_v6
Loaded graph checkpoint from .\checkpoints\dsm_graph_v6.

STAGE 3B: CLEAN GRAPH
source: .\checkpoints\dsm_graph_v6
target: .\checkpoints\dsm_graph_v6_clean
Loaded cleaned graph checkpoint from .\checkpoints\dsm_graph_v6_clean.
Cleanup report: {'original_entities': 3505, 'cleaned_entities': 2578, 'original_relations': 4174, 'cleaned_relations': 3720, 'removed_entities': 27, 'removed_isolated_entities': 900, 'removed_relations': 454}

STAGE 4: COMMUNITIES
checkpoint: .\checkpoints\dsm_graph_v6_clean
GRAPH RAG PIPELINE
Entities:            2578
Relations:           3720
Relation index keys: 2578
Chunk citations:     2047
Communities:         103
Community summaries: 102
Graph checkpoint:    {'checkpoint_version': 1, 'generated_at': '2026-04-09T13:22:43.253412+00:00', 'source_checkpoint_dir': 'checkpoints\\dsm_graph_v6', 'source_meta': {'checkpoint_version': 1, 'generated_at': '2026-04-09T12:21:16.184325+00:00', 'nodes_hash': '9534

2026-04-09 14:39:20.126076: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 14:39:21.524537: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
C:\Users\derrm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can on